In [4]:
import sys
import os

# Add the parent directory one level up to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

# Import the initialize_spark function from the package
from notebooks.config.initialize_spark import *

# Initialize the Spark session and delta table path
spark = initialize_spark()

initialize_spark() executed from: /home/jovyan/work/notebooks/config/initialize_spark.py
DATA_FOLDER being defined as: ../../data
Delta Lake support is not enabled.


In [5]:
from pyspark.sql.functions import explode, col, when

In [6]:
df = (spark.read.format("json")
      .option("multiLine", "true")
      .load(f"{DATA_FOLDER}/nobel_prizes.json"))

df_flattened = (
    df
    .withColumn("laureates",explode(col("laureates")))
    .select(col("category")
            ,col("year")
            ,col("overallMotivation")
            ,col("laureates.id")
            ,col("laureates.firstname")
            ,col("laureates.surname")
            ,col("laureates.share")
            ,col("laureates.motivation")))

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 0) / 1]
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Dropping rows with null values
df_dropna = df_flattened.dropna()

# Displaying the DataFrame after dropping null values
df_dropna.show()

In [ ]:
# Filling null values with a specific value
df_fillna = df_flattened.fillna("N/A")

# Displaying the DataFrame after filling null values
df_fillna.show()


In [ ]:
# Replacing null values based on conditions
df_replace = (
    df_flattened.withColumn("category", when(col("category").isNull(), "").otherwise(col("category")))
    .withColumn("overallMotivation", when(col("overallMotivation").isNull(), "").otherwise(col("overallMotivation")))
    .withColumn("firstname", when(col("firstname").isNull(), "").otherwise(col("firstname")))
    .withColumn("surname", when(col("surname").isNull(), "").otherwise(col("surname")))
    .withColumn("year", when(col("year").isNull(), 9999).otherwise(col("year"))))

# Displaying the DataFrame after replacing null values
df_replace.show()


### Handling null values in user-defined functions (UDFs)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Sample DataFrame with null values
data = [("John", 25), ("Alice", None), ("Bob", 30)]
df = spark.createDataFrame(data, ["name", "age"])

# Define a UDF to handle null values
def process_name(name):
    if name is None:
        return "Unknown"
    else:
        return name.upper()

# Register the UDF
process_name_udf = udf(process_name, StringType())

# Apply the UDF to the DataFrame
df_with_processed_names = df.withColumn("processed_name", process_name_udf(df["name"]))

# Show the resulting DataFrame
df_with_processed_names.show()


### Handling null values in machine learning pipelines

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Imputer

# Create a sample DataFrame with missing values
data = [
    (1, 2.0),
    (2, None),
    (3, 5.0),
    (4, None),
    (5, 7.0)
]
df = spark.createDataFrame(data, ["id", "value"])

# Create an instance of Imputer and specify the input/output columns
imputer = Imputer(inputCols=["value"], outputCols=["imputed_value"])

# Fit the imputer to the data and transform the DataFrame
imputer_model = imputer.fit(df)
imputed_df = imputer_model.transform(df)

# Show the resulting DataFrame
imputed_df.show()

In [7]:
spark.stop()